# Day 4 hands-on: learning uncertainties

A network output without an uncertainty is a measurement without an error bar. In this session we train a regression network that predicts, for every input $x$, both a value $f_\theta(x)$ and its uncertainty $\sigma_\theta(x)$, using the heteroskedastic loss from the lecture,
$$\mathcal{L} = \left\langle \frac{|y - f_\theta(x)|^2}{2\sigma_\theta(x)^2} + \log \sigma_\theta(x) \right\rangle \; .$$
We inject noise of known size into the training data and check whether the network finds it.

**Colab:** everything runs on the CPU, each training takes a few seconds. Run the cells from top to bottom with `Shift+Enter`.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

## 1) A regression task with injected noise

The target is $f(x) = \sin(2\pi x)$ for $x \in [0,1]$. We add Gaussian noise to every training target, $y = f(x) + \sigma(x)\,\epsilon$ with $\epsilon \sim \mathcal{N}(0,1)$, where the width $\sigma(x)$ depends on $x$: the noise is **heteroskedastic**. The network never sees $\sigma(x)$, only the noisy pairs $(x, y)$.

In [ ]:
def f_true(x):
    return torch.sin(2 * math.pi * x)


def sigma_linear(x):
    return 0.05 + 0.3 * x                       # the injected noise grows with x


def make_data(n, sigma_fn, x_min=0.0, x_max=1.0):
    x = x_min + (x_max - x_min) * torch.rand(n, 1)
    y = f_true(x) + sigma_fn(x) * torch.randn_like(x)
    return x, y


x_train, y_train = make_data(5000, sigma_linear)
x_plot = torch.linspace(0, 1, 200)[:, None]

plt.figure(figsize=(6, 4))
plt.plot(x_train, y_train, ".", color="black", ms=2, alpha=0.5, label="training data")
plt.plot(x_plot, f_true(x_plot), color="darkblue", label="$f(x)$")
plt.fill_between(x_plot[:, 0], (f_true(x_plot) - sigma_linear(x_plot))[:, 0],
                 (f_true(x_plot) + sigma_linear(x_plot))[:, 0], color="darkblue", alpha=0.2,
                 label="$f(x) \\pm \\sigma(x)$, injected")
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.legend()
plt.show()

### The usual regression
First the regression from Day 1: a network with one output, trained with the MSE. Nothing to do here, just run it. It learns the mean well, but it has no way to tell us that the data at $x \approx 1$ is much noisier than at $x \approx 0$.

In [ ]:
class Regressor(nn.Module):
    def __init__(self, n_outputs=1, n_hidden=64):
        super().__init__()
        self.layer1 = nn.Linear(1, n_hidden)
        self.layer2 = nn.Linear(n_hidden, n_hidden)
        self.layer3 = nn.Linear(n_hidden, n_hidden)
        self.layer4 = nn.Linear(n_hidden, n_outputs)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = torch.relu(self.layer3(x))
        return self.layer4(x)


def train(model, x, y, loss_function, n_epochs=200, lr=1e-3):
    loader = DataLoader(TensorDataset(x, y), batch_size=256, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)   # smaller steps towards the end
    for epoch in range(n_epochs):
        for x_batch, y_batch in loader:
            loss = loss_function(model(x_batch), y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
    print(f"final loss {loss.item():.3f}")


def mse_loss(output, y):
    return ((output - y)**2).mean()


mse_model = Regressor(n_outputs=1)
train(mse_model, x_train, y_train, mse_loss)

with torch.no_grad():
    f_mse = mse_model(x_plot)
plt.figure(figsize=(6, 4))
plt.plot(x_train, y_train, ".", color="black", ms=2, alpha=0.3, label="training data")
plt.plot(x_plot, f_true(x_plot), color="darkblue", label="$f(x)$")
plt.plot(x_plot, f_mse, color="darkred", ls="--", label="MSE network")
plt.xlabel("$x$")
plt.legend()
plt.show()

## 2) Task 1: the heteroskedastic loss

We give the network a second output. The first output is the prediction $f_\theta(x)$, the second is $\log \sigma_\theta(x)$: predicting the logarithm guarantees $\sigma_\theta = e^{\log \sigma_\theta} > 0$ for any network output.

**Task:** implement the heteroskedastic loss, the negative logarithm of a Gaussian likelihood with learned mean and width. Then train and compare the learned $\sigma_\theta(x)$ with the injected $\sigma(x)$.

In [ ]:
def heteroskedastic_loss(output, y):
    f_pred, log_sigma = output[:, :1], output[:, 1:]
    ########################################
    ### Implement the loss here ###
    loss = ...
    ########################################
    return loss.mean()

In [ ]:
def predict(model, x):
    with torch.no_grad():
        output = model(x)
    return output[:, :1], output[:, 1:].exp()          # f_theta(x), sigma_theta(x)


def plot_prediction(model, sigma_fn, x_train, y_train, x_range=(0, 1)):
    x = torch.linspace(*x_range, 300)[:, None]
    f_pred, sigma_pred = predict(model, x)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.plot(x_train, y_train, ".", color="black", ms=2, alpha=0.3, label="training data")
    ax.plot(x, f_true(x), color="darkblue", label="$f(x)$")
    ax.plot(x, f_pred, color="darkred", ls="--", label="$f_\\theta(x)$")
    ax.fill_between(x[:, 0], (f_pred - sigma_pred)[:, 0], (f_pred + sigma_pred)[:, 0],
                    color="darkred", alpha=0.25, label="$f_\\theta \\pm \\sigma_\\theta$")
    ax.set_xlabel("$x$")
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=8)
    ax = axes[1]
    inside = (x[:, 0] >= 0) & (x[:, 0] <= 1)                  # the noise is only injected where we have data
    ax.plot(x[inside], (sigma_fn(x) * torch.ones_like(x))[inside], color="darkblue", label="injected $\\sigma(x)$")
    ax.axvspan(0, 1, color="grey", alpha=0.1)
    ax.plot(x, sigma_pred, color="darkred", ls="--", label="learned $\\sigma_\\theta(x)$")
    ax.set_xlabel("$x$")
    ax.set_ylim(0, None)
    ax.legend(fontsize=8)
    plt.show()


het_model = Regressor(n_outputs=2)
train(het_model, x_train, y_train, heteroskedastic_loss)
plot_prediction(het_model, sigma_linear, x_train, y_train)

*Discussion in class.*

## 3) Task 2: does the uncertainty follow the noise?

**Task:** inject a different noise pattern and check that the learned $\sigma_\theta(x)$ follows it. Some ideas are listed in the cell, but invent your own. What happens where the noise is zero?

In [ ]:
def sigma_new(x):
    ########################################
    ### Define the injected noise here ###
    return 0.05 + 0.3 * torch.exp(-((x - 0.5) / 0.1)**2)     # a bump in the middle
    # return 0.1 * torch.ones_like(x)                        # constant noise
    # return 0.05 + 0.25 * (x > 0.5).float()                 # a step
    # return 0.3 * torch.abs(torch.sin(4 * math.pi * x))     # oscillating, with zeros
    ########################################


x_new, y_new = make_data(5000, sigma_new)
het_new = Regressor(n_outputs=2)
train(het_new, x_new, y_new, heteroskedastic_loss)
plot_prediction(het_new, sigma_new, x_new, y_new)

*Discussion in class.*

## 4) Task 3: are the uncertainties correct? Pulls

In physics we test error bars with **pulls**: for independent test points,
$$\text{pull} = \frac{y - f_\theta(x)}{\sigma_\theta(x)}$$
should follow a standard Gaussian, mean 0 and width 1, if the prediction is unbiased and the uncertainties are right. A width above 1 means underestimated uncertainties, below 1 overestimated ones.

**Task:** compute the pulls of the heteroskedastic network from Task 1 on a fresh test set. For comparison, the cell also computes the pulls of the MSE network with a single constant uncertainty, the root mean square of its residuals. Compare the two distributions, and look at the pulls as a function of $x$.

In [ ]:
x_test, y_test = make_data(5000, sigma_linear)
f_het, sigma_het = predict(het_model, x_test)

########################################
### Compute the pulls here ###
pulls_het = ...
########################################

with torch.no_grad():
    f_mse_test = mse_model(x_test)
sigma_const = ((y_test - f_mse_test)**2).mean().sqrt()          # one uncertainty for all x
pulls_const = ((y_test - f_mse_test) / sigma_const)[:, 0]

bins = np.linspace(-5, 5, 51)
z = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ax = axes[0]
ax.hist(pulls_het, bins, density=True, histtype="step", color="darkred", lw=2,
        label=f"heteroskedastic: mean {pulls_het.mean():.2f}, width {pulls_het.std():.2f}")
ax.hist(pulls_const, bins, density=True, histtype="step", color="grey", lw=2,
        label=f"constant $\\sigma$: mean {pulls_const.mean():.2f}, width {pulls_const.std():.2f}")
ax.plot(z, np.exp(-z**2 / 2) / np.sqrt(2 * np.pi), color="black", ls=":", label="$\\mathcal{N}(0,1)$")
ax.set_xlabel("pull")
ax.legend(fontsize=8)
for ax, pulls, name, color in [(axes[1], pulls_const, "constant $\\sigma$", "grey"),
                               (axes[2], pulls_het, "heteroskedastic", "darkred")]:
    ax.plot(x_test, pulls, ".", color=color, ms=2, alpha=0.5)
    ax.axhline(2, color="black", ls=":", lw=1)
    ax.axhline(-2, color="black", ls=":", lw=1)
    ax.set_title(f"pulls, {name}")
    ax.set_xlabel("$x$")
    ax.set_ylim(-5, 5)
plt.show()

*Discussion in class.*

## 5) Task 4: what the network does not know

Our training data covers $x \in [0,1]$. **Task:** plot the prediction of the heteroskedastic network for $x \in [-0.3, 1.3]$. Does the learned uncertainty warn us when we leave the training data?

In [ ]:
########################################
### Choose the x-range here ###
x_range = (-0.3, 1.3)
########################################
plot_prediction(het_model, sigma_linear, x_train, y_train, x_range=x_range)

*Discussion in class.*

## 6) Bonus: sampling over networks

In the lecture we saw that a Bayesian network works much like an ensemble of networks trained on the same data. An ensemble is easy to build: train the same network several times with different random initializations. The spread of the predictions, $\sigma_\text{stat}$, measures how well the data determines the network, and combines with the learned noise $\sigma_\text{syst}$ as
$$\sigma_\text{tot}^2 = \sigma_\text{syst}^2 + \sigma_\text{stat}^2 \; .$$

**Bonus Task:** train an ensemble of 5 heteroskedastic networks, and plot $\sigma_\text{syst}$, $\sigma_\text{stat}$ and $\sigma_\text{tot}$ for $x \in [-0.3, 1.3]$, together with the actual error $|f_\theta(x) - f(x)|$ of the ensemble mean. Where does $\sigma_\text{stat}$ matter?

In [ ]:
#################
# Add code here #
#################